In [60]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from scipy.stats import multivariate_normal, norm

snp = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "snp500.csv", index_col=1)
china = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "china_transformed.csv", index_col=1).reset_index()
em = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "emerging_markets.csv", index_col=1)
gold = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "gold.csv", index_col=1)
india = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "india.csv", index_col=1)
mscieurope = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "msci_europe.csv", index_col=1)
smallcapeurope = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "small_cap_europe.csv", index_col=1)

labels = ["snp", "china", "em", "gold", "india", "mscieurope", "smallcapeurope"]
dfs = [snp, china, em, gold, india, mscieurope, smallcapeurope]

def transform_date(date):
    parts = date.split(' ')
    if len(parts) == 1:
        return date
    return parts[0]

for df in dfs:
    df["Date"] = df["Date"].apply(transform_date)

china

date_sets = [set(df["Date"]) for df in dfs]

# Find the intersection of all date sets
common_dates = set.intersection(*date_sets)

# Filter each DataFrame to include only common dates
dfs = [df[df["Date"].isin(common_dates)] for df in dfs]

N = len(dfs)
T = len(dfs[0]) - 1

returns_all = np.empty((T, N))
sorted_returns_all = np.empty((T, N))
for i, df in enumerate(dfs):
    returns = df["Close"].pct_change().values[1:]
    returns_all[:, i] = returns
    sorted_returns_all[:, i] = sorted(returns)

all_dates = dfs[0]["Date"].values[1:]

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Projects\\portfolio\\data\\OHCL\\china_transformed.csv'

In [23]:
def perform_validation(w, validation_data):
    # validation data is shape (T, N)
    
    # perform metrics
    total_return = validation_data @ w # shape (T,)
    cumulative_returns = np.cumprod(total_return + 1) - 1
    risk_free = 0.0
    std = np.std(total_return)
    downside_std = np.std(total_return[total_return > 0])
    sharpe = (255 * np.mean(total_return) - risk_free) / (np.sqrt(255) * std)
    sortino = (255 * np.mean(total_return) - risk_free) / (np.sqrt(255) * downside_std)    

    alpha = 0.10

    var_alpha = np.quantile(total_return, alpha)
    cvar = total_return[total_return <= var_alpha].mean()
    mean_return = np.mean(total_return)
    return var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return
    

In [24]:
def lower_part_mean(matrix):
    n, _ = matrix.shape
    s = 0.0
    for i in range(1,n):
        for j in range(i):
            s += matrix[i,j]
    N = (n-1)*n/2
    return s / N

def compute_statistics_rolling(returns, window_size, stepsize):
    all_corrs = []
    all_volas = []
    T, N = returns.shape
    w = np.ones(N) / N
    for idx in range(0, T - window_size, stepsize):
        section = returns[idx:idx+window_size,:]
        corr = np.corrcoef(section, rowvar=False) # (N,N)
        all_corrs.append(lower_part_mean(corr))
        returns_total = section @ w
        std = np.std(returns_total)
        all_volas.append(std)
    
    all_corrs = np.array(all_corrs)
    all_volas = np.array(all_volas)
    return all_corrs, all_volas


In [ ]:
import tensorflow as tf

# train weights
alpha = 0.05   # tail probability for CVaR
lambda_ = 0.5  # CVaR penalty
learning_rate = 0.01
gamma = 0.02
num_steps = 2000
beta = 0.1 # penalty volatility

def gradient(train_data, corr_thresh, vol_thresh):

    # returns (weights, mode)
    # where mode = 0 if returned normal (optimal) weights,
    # 1 = if returned equal because of correlations
    # 2 = if returned equal because of volatility

    # train data: (T,N)
    w = tf.Variable(tf.ones([N], dtype=tf.float32) / N)
    t = tf.Variable(0.0, dtype=tf.float32)  # VaR threshold

    optimizer = tf.keras.optimizers.Adam(learning_rate)

    X_tf = tf.convert_to_tensor(train_data, dtype=tf.float32) # (T,N)
    #cov_tf = tf.convert_to_tensor(cov, dtype=tf.float32)  # shape (N, N)

    corr = np.corrcoef(train_data, rowvar=False)
    mean_corr = lower_part_mean(corr)

    if mean_corr > corr_thresh:
        return np.ones(N) / N, 1

    for step in range(num_steps):
        with tf.GradientTape() as tape:
            # enforce sum-to-1 via softmax
            w_normalized = tf.nn.softmax(w)
            w_col = tf.reshape(w_normalized, [N, 1])  # shape (N,1)

            # portfolio returns
            portfolio_returns = tf.matmul(X_tf, tf.reshape(w_col, [-1,1])) # (T)
            losses = -portfolio_returns
            cvar = t + (1 / (1 - alpha)) * tf.reduce_mean(tf.nn.relu(losses - t))

            objective = -(tf.reduce_mean(portfolio_returns) - lambda_ * cvar) + gamma*tf.reduce_sum(tf.math.square(w_col))

        # compute gradients
        grads = tape.gradient(objective, [w, t])
        optimizer.apply_gradients(zip(grads, [w, t]))

        # if step % 200 == 0:
        #     print(f"Step {step}: objective = {-objective.numpy():.6f}")

    # after the optimization loop
    optimal_w = tf.nn.softmax(w).numpy()

    # compute volatility
    combined_returns = train_data @ optimal_w # (T,)
    volatility = np.std(combined_returns)

    print('volatility:', volatility)

    if volatility > vol_thresh:
        return np.ones(N) / N, 2

    return optimal_w, 0

In [26]:
def rolling_window(window_size, stepsize, validation_size, data, corr_thresh, vol_thresh):
    # data is (T,N)

    sharpes_w_opt= []
    sharpes_w_eqw = []

    mean_return_w_opt = []
    mean_return_w_eqw = []

    sortinos_w_opt = []
    sortinos_w_eqw = []

    vars_w_opt = []
    vars_w_eqw = []

    cvars_w_opt = []
    cvars_w_eqw = []

    T, N = data.shape
    weights_opt = []

    mode_counter = [0, 0, 0] # normal, corr, vol

    for idx in range(0, T - window_size - validation_size, stepsize):
        train_data = data[idx:idx+window_size]
        val_data = data[idx+window_size:idx+window_size+validation_size]
        cov = np.cov(train_data, rowvar=False)

        w, mode = gradient(train_data, corr_thresh, vol_thresh)
        weights_opt.append(w)
        mode_counter[mode] += 1

        (
            var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return
        ) = perform_validation(w, val_data)

        sharpes_w_opt.append(sharpe)
        sortinos_w_opt.append(sortino)
        vars_w_opt.append(var_alpha)
        cvars_w_opt.append(cvar)
        mean_return_w_opt.append(mean_return)

        (
            var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return
        ) = perform_validation(np.ones(N) / N, val_data)

        sharpes_w_eqw.append(sharpe)
        sortinos_w_eqw.append(sortino)
        vars_w_eqw.append(var_alpha)
        cvars_w_eqw.append(cvar)
        mean_return_w_eqw.append(mean_return)

    df = pd.DataFrame({
        "Sharpes": [np.mean(sharpes_w_eqw), np.mean(sharpes_w_opt)],
        "Sortinos": [np.mean(sortinos_w_eqw), np.mean(sortinos_w_opt)],
        "Vars": [np.mean(vars_w_eqw), np.mean(vars_w_opt)],
        "Cvars": [np.mean(cvars_w_eqw), np.mean(cvars_w_opt)],
        "MeanReturn": [np.mean(mean_return_w_eqw), np.mean(mean_return_w_opt)],
    }, index=["Equal weights", "Return + CVaR"])

    weights_opt = np.array(weights_opt)

    return df, np.mean(weights_opt, axis=0), mode_counter

In [35]:
# run

corrs, volas = compute_statistics_rolling(returns_all, 90, 30)

print('mean corrs:', np.mean(corrs))
print('mean volas:', np.mean(volas))
print('highest corrs:', np.max(corrs))
print('lowest corrs:', np.min(corrs))
print('lowest volas:', np.min(volas))
print('highest volas:', np.max(volas))
# results:
# based on this, determine threshold for correlation and volatility

vola_thresh = np.percentile(volas, 75)
corr_thresh = np.percentile(corrs, 75)

mean corrs: 0.27776446045236847
mean volas: 0.007112654453718704
highest corrs: 0.5411403257539809
lowest corrs: 0.12343539983118701
lowest volas: 0.003985026483839505
highest volas: 0.020371210003547535


In [36]:
df, weights_opt, mode_counter = rolling_window(
    window_size=252*2,
    stepsize=50,
    validation_size=252,
    data=returns_all,
    corr_thresh=corr_thresh,
    vol_thresh=vola_thresh
)
df

print('mean weights optimized:', weights_opt)
s = sum(mode_counter)
print('pct of weights chosen optimally:', mode_counter[0]/s)
print('pct of weights chosen bec. of correlation:', mode_counter[1]/s)
print('pct of weights chosen bec. of volatility:', mode_counter[2]/s)


mean corr: 0.3146895224437617
volatility: 0.008011543040477836
mean corr: 0.3192683862519139
volatility: 0.008031807131751726
mean corr: 0.285347320074745
volatility: 0.007089430489511603
mean corr: 0.2671036610566055
volatility: 0.006807868435583214
mean corr: 0.24499549814408886
volatility: 0.006395572854578154
mean corr: 0.24340507965507868
volatility: 0.005943643011084967
mean corr: 0.2674312130216749
volatility: 0.005790624274016028
mean corr: 0.2714532419218241
volatility: 0.0059412993328229495
mean corr: 0.26333396643608775
volatility: 0.005769283103608528
mean corr: 0.27237585972744255
volatility: 0.00589663870837264
mean corr: 0.2702437872257781
volatility: 0.005878117165744408
mean corr: 0.25917833100183285
volatility: 0.0058144048825610455
mean corr: 0.26027825602171906
volatility: 0.005777793557706755
mean corr: 0.35469840250372203
mean corr: 0.4019286776639563
mean corr: 0.3948457869661956
mean corr: 0.3997618862889433
mean corr: 0.3927351369859116
mean corr: 0.39535067916

In [38]:
print(weights_opt)
print()
df

[0.14874006 0.13657343 0.13837568 0.14790305 0.14572448 0.14129864
 0.14138465]



,Sharpes,Sortinos,Vars,Cvars,MeanReturn
Equal weights,0.920070,1.480642,-0.007855,-0.013596,0.000374
Return + CVaR,0.924078,1.495343,-0.007795,-0.013501,0.000370


Stress test bear markets

In [ ]:



def stress_test(df_data, start_train, end_train, end_valid):

    val_returns_total = []
    train_returns_total = []

    modes = np.zeros(3)

    for df in df_data:
        df_datetime = df.copy()
        df_datetime["Date"] = pd.to_datetime(df_datetime["Date"])
        train = df_datetime[
            (df_datetime["Date"] >= pd.to_datetime(start_train))
            & (df_datetime["Date"] <= pd.to_datetime(end_train))]
        val = df_datetime[
            (df_datetime["Date"] >= pd.to_datetime(end_train))
            & (df_datetime["Date"] <= pd.to_datetime(end_valid))
        ]

        train_returns = train["Close"].pct_change().values[1:]
        val_returns = val["Close"].pct_change().values[1:]

        val_returns_total.append(val_returns)
        train_returns_total.append(train_returns)
        
    train_returns_total = np.array(train_returns_total).T # shape (num_days, N = num_assets)
    val_returns_total = np.array(val_returns_total).T

    w_opt, mode_idx = gradient(train_returns_total, corr_thresh, vola_thresh)
    print('chosen weights optimal:', w_opt)
    modes[mode_idx] += 1
    
    return perform_validation(w_opt, val_returns_total), perform_validation(np.ones(N) / N, val_returns_total), modes


In [ ]:
start_train = "2024-01-01"
end_train = "2025-01-01"
end_valid = "2025-04-01"

results_optimized, results_eqw, modes = stress_test(
    dfs, start_train, end_train, end_valid
)
var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return = results_optimized

print('var:', var_alpha)
print('cvar:', cvar)
print('sharpe:', sharpe)
print('sortino:', sortino)
print('mean return:', mean_return)
print()

var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return = results_eqw

print('var:', var_alpha)
print('cvar:', cvar)
print('sharpe:', sharpe)
print('sortino:', sortino)
print('mean return:', mean_return)
print()

s = sum(mode_counter)
print('pct of weights chosen optimally:', mode_counter[0]/s)
print('pct of weights chosen bec. of correlation:', mode_counter[1]/s)
print('pct of weights chosen bec. of volatility:', mode_counter[2]/s)


fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(len(cumulative_returns))), y=cumulative_returns))
fig.show()

mean corr: 0.23971550050792625
volatility: 0.005443921979458313
chosen weights optimal: [0.17072469 0.10815918 0.1384753  0.17181906 0.14737406 0.13307038
 0.13037725]
var: -0.007755963953030006
cvar: -0.010639804910435566
sharpe: 0.5820740457672601
sortino: 1.2082771395272243
mean return: 0.00021372179524967017

pct of weights chosen optimally: 0.6578947368421053
pct of weights chosen bec. of correlation: 0.2894736842105263
pct of weights chosen bec. of volatility: 0.05263157894736842


In [52]:
start_train = "2019-02-05"
end_train = "2020-02-12"
end_valid = "2020-04-03"

central, modes = stress_test(
    dfs, start_train, end_train, end_valid
)
var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return = central

print('var:', var_alpha)
print('cvar:', cvar)
print('sharpe:', sharpe)
print('sortino:', sortino)
print('mean return:', mean_return)
print()

s = sum(mode_counter)
print('pct of weights chosen optimally:', mode_counter[0]/s)
print('pct of weights chosen bec. of correlation:', mode_counter[1]/s)
print('pct of weights chosen bec. of volatility:', mode_counter[2]/s)


fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(len(cumulative_returns))), y=cumulative_returns))
fig.show()

mean corr: 0.22960352816685634
volatility: 0.005273138309151943
chosen weights optimal: [0.1651642  0.09643536 0.1311008  0.16436002 0.14088394 0.14744465
 0.15461107]
var: -0.03875047850396306
cvar: -0.05698083804525964
sharpe: -3.5738629181121055
sortino: -7.084714691931583
mean return: -0.006044814051279823

pct of weights chosen optimally: 0.6578947368421053
pct of weights chosen bec. of correlation: 0.2894736842105263
pct of weights chosen bec. of volatility: 0.05263157894736842


Optimize specifically for bear markets

In [58]:
train_returns_all = []
val_returns_all = []
for df in dfs:
    df_datetime = df.copy()
    df_datetime["Date"] = pd.to_datetime(df_datetime["Date"])
    train = df_datetime[
        (df_datetime["Date"] >= pd.to_datetime("2021-12-28"))
        & (df_datetime["Date"] <= pd.to_datetime("2022-10-10"))]
    val = df_datetime[
        (df_datetime["Date"] >= pd.to_datetime("2025-02-13"))
        & (df_datetime["Date"] <= pd.to_datetime("2025-04-23"))
    ]

    train_returns = train["Close"].pct_change().values[1:]
    val_returns = val["Close"].pct_change().values[1:]

    train_returns_all.append(train_returns)
    val_returns_all.append(val_returns)

train_returns_all = np.array(train_returns_all).T
val_returns_all = np.array(val_returns_all).T

w_opt, mode_idx = gradient(train_returns_all, np.inf, np.inf)
    
(
    var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return
) = perform_validation(w_opt, val_returns_all)

print('OPTIMIZED WEIGHTS:')
print('var:', var_alpha)
print('cvar:', cvar)
print('sharpe:', sharpe)
print('sortino:', sortino)
print('mean return:', mean_return)
print()

(
    var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return
) = perform_validation(np.ones(N) / N, val_returns_all)

print('EQUAL WEIGHTS:')
print('var:', var_alpha)
print('cvar:', cvar)
print('sharpe:', sharpe)
print('sortino:', sortino)
print('mean return:', mean_return)
print()


mean corr: 0.2637707357867618
volatility: 0.007709847191024051
OPTIMIZED WEIGHTS:
var: -0.010878626276888561
cvar: -0.019826607302745644
sharpe: -0.49653561857733264
sortino: -0.8199460717225051
mean return: -0.00029525775633987647

EQUAL WEIGHTS:
var: -0.011568444233122054
cvar: -0.023371565606841595
sharpe: -0.5273990135028637
sortino: -0.8830486498654954
mean return: -0.0003696206881158322



In [59]:
print(w_opt)

[0.13744248 0.22792344 0.09481022 0.18408944 0.16422191 0.09244009
 0.09907243]
